In [1]:
import os
from tqdm import tqdm
import sys
if not hasattr(sys.modules[__name__], "cwd_changed"):
    os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(__name__))))
    sys.modules[__name__].cwd_changed = True

import warnings 
warnings.filterwarnings("ignore")
import logging
logging.getLogger('pgmpy').setLevel(logging.WARNING)



import pandas as pd
from utils.graph import dag_to_cpdag
import ast
from metrics.graph import compare_dags
from utils.results import *
from utils.plotting import *

In [2]:
import pickle

subset = "jpmf_grid"
with open(f'results/ea_fes/{subset}.pkl', 'rb') as f:
    results_ea_fes = pickle.load(f)

In [3]:
from pgmpy.base import DAG as _pgmpy_DAG
from pgmpy.estimators import BIC, HillClimbSearch
import networkx as nx

def final_greedy_refine(data, g_best: nx.DiGraph) -> nx.DiGraph:
    """
    Run a last greedy hill-climb starting from GA's best graph.
    Uses pgmpy HillClimbSearch when self.scorer is a pgmpy StructureScore.
    """
    # Convert nx -> pgmpy DAG
    start = _pgmpy_DAG()
    start.add_nodes_from(data.columns)
    start.add_edges_from(list(g_best.edges()))

    est = HillClimbSearch(data)
    scorer = BIC(data)

    # HillClimbSearch supports start_dag + max_indegree etc. :contentReference[oaicite:0]{index=0}
    greedy_epsilon: float = 1e-4,
    
    # for i in range
    refined = est.estimate(
        scoring_method="bic-d",      # your BIC/BDeu/BICSynergy instance
        start_dag=start,
        # max_indegree=self.max_parents,
        epsilon=greedy_epsilon,
        show_progress=False,
    )
    best_score = scorer.score(refined)
    return nx.DiGraph(refined.edges()), best_score

In [4]:

from utils.data import read_data_files


data, metadata = read_data_files(f"experiments/RQ1/{subset}")

In [5]:
df_by_id = {key: pd.read_csv(f"experiments/RQ1/{subset}/{value}") for key, value in data.items()}
metadata_by_id = {key: pd.read_csv(f"experiments/RQ1/{subset}/{value}") for key, value in metadata.items()}

In [ ]:
# from concurrent.futures import ThreadPoolExecutor



# with ThreadPoolExecutor() as executor:
#     futures = {executor.submit(refine_dag, key, value): key for key, value in results_ea_fes[0].learned_dags.items()}
#     for future in futures:
#         key, refined, refined_score = future.result()
#         refined_dag[key] = refined


In [12]:
import pickle

subset = 'jpmf_grid'
# subset = 'toyMedium'

with open(f'results/pc/{subset}.pkl', 'rb') as f:
    results_pc = pickle.load(f)
with open(f'results/ges/{subset}.pkl', 'rb') as f:
    results_ges = pickle.load(f)
with open(f'results/hc/{subset}.pkl', 'rb') as f:
    results_hc = pickle.load(f)

with open(f'results/ea_ues/{subset}.pkl', 'rb') as f:
    results_ea_ues = pickle.load(f)
with open(f'results/ea_ies/{subset}.pkl', 'rb') as f:
    results_ea_ies = pickle.load(f)
with open(f'results/ea_fes/{subset}.pkl', 'rb') as f:
    results_ea_fes = pickle.load(f)

# with open(f'results/ea_upu/{subset}.pkl', 'rb') as f:
#     results_ea_upu = pickle.load(f)
with open(f'results/ea_ipu/{subset}.pkl', 'rb') as f:
    results_ea_ipu = pickle.load(f)

results_by_label = {"PC": results_pc, "GES": results_ges, "HC": results_hc, 
 "EA Uninformed": results_ea_ues,
    "EA Informed": results_ea_ies,
    "EA Fully Informed": results_ea_fes,
    # "EA Uninformed (PU)": results_ea_upu,
    "EA Informed (PU)": results_ea_ipu,

    }

mapping="experiments/RQ1/jpmf_grid/grid_mapping.csv"

x_col="target_mi"
y_col="syn_cutoff"

In [13]:
from metrics.graph import evaluate_colliders
from concurrent.futures import ThreadPoolExecutor

results_ea_final = []

for each_rep in results_ea_fes:
    
    learned_dags = each_rep.learned_dags
    true_dags = each_rep.true_dags

    print(learned_dags)
    for key, value in learned_dags.items():
        learned_dags[key] = final_greedy_refine(df_by_id[key], value)
    for key, value in true_dags.items():
        true_dags[key] = final_greedy_refine(df_by_id[key], value)

    # learned_dags = learned_dags
    # each_rep.true_dags = true_dags
    eval_metrics = {}
    for idx in range(len(true_dags)):
        eval_metrics[idx] = compare_dags(true_dag=true_dags[idx], learned_dag=learned_dags[idx])
        eval_metrics[idx].update(evaluate_colliders(metadata_df=each_rep.metadata[idx], learned_dag=learned_dags[idx]))
    eval_metrics = pd.DataFrame(eval_metrics).T

    greedy_last_step = {
        "learned_dags": learned_dags,
        "true_dags": true_dags,
        "metadata": each_rep.metadata,
        "eval_metrics": eval_metrics
    }
    results_ea_final.append(greedy_last_step)

{12: <networkx.classes.digraph.DiGraph object at 0x000002837060C860>, 14: <networkx.classes.digraph.DiGraph object at 0x000002837060CCB0>, 0: <networkx.classes.digraph.DiGraph object at 0x000002837060D100>, 11: <networkx.classes.digraph.DiGraph object at 0x000002837060D550>, 16: <networkx.classes.digraph.DiGraph object at 0x000002837060D9A0>, 17: <networkx.classes.digraph.DiGraph object at 0x000002837060DDF0>, 10: <networkx.classes.digraph.DiGraph object at 0x000002837060E240>, 13: <networkx.classes.digraph.DiGraph object at 0x000002837060E690>, 1: <networkx.classes.digraph.DiGraph object at 0x000002837060EAE0>, 19: <networkx.classes.digraph.DiGraph object at 0x000002837060EF30>, 15: <networkx.classes.digraph.DiGraph object at 0x000002837060F380>, 18: <networkx.classes.digraph.DiGraph object at 0x000002837060F7D0>, 20: <networkx.classes.digraph.DiGraph object at 0x000002837060FC20>, 22: <networkx.classes.digraph.DiGraph object at 0x00000283705880B0>, 2: <networkx.classes.digraph.DiGrap

ValueError: 'start_dag' should be a DAG with the same variables as the data set, or 'None'.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

# --- UI ---
metric_dropdown = widgets.Dropdown(
    options=["Recall (Collider)", "F1 (Collider)", "SHD", "Recall (Synergy)"],
    value="Recall (Collider)",
    description="Metric:",
)

out = widgets.Output()

def render(metric: str):
    """Render the heatmap for the selected metric."""
    with out:
        clear_output(wait=True)

        # Important: close prior figures to avoid stacking / memory leaks
        plt.close("all")

        fig, axes = plot_final_heatmap(
            results_by_label,
            mapping=mapping,
            subset_metrics=[metric],
            plot_metric=metric,
            x_col=x_col,
            y_col=y_col,
        )

        # Ensure the latest figure is shown in the output area
        plt.show()

def on_metric_change(change):
    if change["name"] == "value":
        render(change["new"])

metric_dropdown.observe(on_metric_change, names="value")

# Display UI + initial render
display(widgets.VBox([metric_dropdown, out]))
render(metric_dropdown.value)
